In [4]:
import pandas as pd

# Load dataset
train_df = pd.read_csv("/content/phm_train.csv")
test_df = pd.read_csv("/content/phm_test.csv")

print(train_df.head())
print(train_df['label'].value_counts())

       tweet_id  label                                              tweet
0  6.430000e+17      0  user_mention all i can tell you is i have had ...
1  6.440000e+17      0  my doctor told me stop he gave me sum pop i mi...
2  8.150000e+17      1  i take tylenol and i wake up in the middle of ...
3  6.820000e+17      0  i got xans in an advil bottle i dont take them...
4  6.440000e+17      1  mom says i need to stop eating so much bc ive ...
label
0    7091
1    2900
Name: count, dtype: int64


In [5]:
import re

def clean_tweet(text):
    text = text.lower()

    # Remove URLs
    text = re.sub(r'http\S+|www\S+', '', text)

    # Remove mentions
    text = re.sub(r'@\w+', '', text)

    # Remove hashtags symbol only
    text = re.sub(r'#', '', text)

    # Remove special characters
    text = re.sub(r'[^a-zA-Z\s]', '', text)

    # Remove extra spaces
    text = re.sub(r'\s+', ' ', text).strip()

    return text

train_df['clean_text'] = train_df['tweet'].apply(clean_tweet)
test_df['clean_text'] = test_df['tweet'].apply(clean_tweet)

In [7]:
import nltk

nltk.download('punkt')
nltk.download('punkt_tab')

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!
[nltk_data] Downloading package punkt_tab to /root/nltk_data...
[nltk_data]   Unzipping tokenizers/punkt_tab.zip.


True

In [8]:
from nltk.tokenize import word_tokenize
import nltk
nltk.download('punkt')

train_df['tokens'] = train_df['clean_text'].apply(word_tokenize)
test_df['tokens'] = test_df['clean_text'].apply(word_tokenize)

[nltk_data] Downloading package punkt to /root/nltk_data...
[nltk_data]   Package punkt is already up-to-date!


In [9]:
from collections import Counter

# Build vocabulary
counter = Counter()

for tokens in train_df['tokens']:
    counter.update(tokens)

# Minimum frequency threshold
min_freq = 2

vocab = {word for word, freq in counter.items() if freq >= min_freq}

# Add special tokens
word2idx = {'<PAD>': 0, '<UNK>': 1}

for word in vocab:
    word2idx[word] = len(word2idx)

idx2word = {idx: word for word, idx in word2idx.items()}

vocab_size = len(word2idx)
print("Vocabulary size:", vocab_size)

Vocabulary size: 5442


In [10]:
def encode(tokens, word2idx):
    return [word2idx.get(word, word2idx['<UNK>']) for word in tokens]

train_df['encoded'] = train_df['tokens'].apply(lambda x: encode(x, word2idx))
test_df['encoded'] = test_df['tokens'].apply(lambda x: encode(x, word2idx))

In [11]:
from torch.nn.utils.rnn import pad_sequence
import torch

MAX_LEN = 50

def pad_sequence_custom(seq, max_len):
    if len(seq) > max_len:
        return seq[:max_len]
    else:
        return seq + [0] * (max_len - len(seq))

train_padded = [pad_sequence_custom(seq, MAX_LEN) for seq in train_df['encoded']]
test_padded = [pad_sequence_custom(seq, MAX_LEN) for seq in test_df['encoded']]

X_train = torch.tensor(train_padded)
X_test = torch.tensor(test_padded)

y_train = torch.tensor(train_df['label'].values, dtype=torch.float32)
y_test = torch.tensor(test_df['label'].values, dtype=torch.float32)

In [13]:
!wget http://nlp.stanford.edu/data/glove.twitter.27B.zip
!unzip glove.twitter.27B.zip

--2026-02-28 07:36:48--  http://nlp.stanford.edu/data/glove.twitter.27B.zip
Resolving nlp.stanford.edu (nlp.stanford.edu)... 171.64.67.140
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:80... connected.
HTTP request sent, awaiting response... 302 Found
Location: https://nlp.stanford.edu/data/glove.twitter.27B.zip [following]
--2026-02-28 07:36:48--  https://nlp.stanford.edu/data/glove.twitter.27B.zip
Connecting to nlp.stanford.edu (nlp.stanford.edu)|171.64.67.140|:443... connected.
HTTP request sent, awaiting response... 301 Moved Permanently
Location: https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip [following]
--2026-02-28 07:36:48--  https://downloads.cs.stanford.edu/nlp/data/glove.twitter.27B.zip
Resolving downloads.cs.stanford.edu (downloads.cs.stanford.edu)... 171.64.64.22
Connecting to downloads.cs.stanford.edu (downloads.cs.stanford.edu)|171.64.64.22|:443... connected.
HTTP request sent, awaiting response... 200 OK
Length: 1520408563 (1.4G) [ap

In [14]:
import numpy as np

embedding_dim = 200

# Load GloVe
glove_path = "glove.twitter.27B.200d.txt"

embeddings_index = {}

with open(glove_path, 'r', encoding='utf8') as f:
    for line in f:
        values = line.split()
        word = values[0]
        vector = np.asarray(values[1:], dtype='float32')
        embeddings_index[word] = vector

print("Loaded GloVe vectors:", len(embeddings_index))

Loaded GloVe vectors: 1193514


In [15]:
embedding_matrix = np.zeros((vocab_size, embedding_dim))

for word, idx in word2idx.items():
    vector = embeddings_index.get(word)
    if vector is not None:
        embedding_matrix[idx] = vector
    else:
        embedding_matrix[idx] = np.random.normal(scale=0.6, size=(embedding_dim,))

embedding_matrix = torch.tensor(embedding_matrix, dtype=torch.float32)

In [16]:
from torch.utils.data import Dataset, DataLoader

class TweetDataset(Dataset):
    def __init__(self, X, y):
        self.X = X
        self.y = y

    def __len__(self):
        return len(self.X)

    def __getitem__(self, idx):
        return self.X[idx], self.y[idx]

batch_size = 32

train_dataset = TweetDataset(X_train, y_train)
test_dataset = TweetDataset(X_test, y_test)

train_loader = DataLoader(train_dataset, batch_size=batch_size, shuffle=True)
test_loader = DataLoader(test_dataset, batch_size=batch_size)

In [38]:
def train_model(model, train_loader, test_loader, epochs=10):

    for epoch in range(epochs):
        model.train()
        total_loss = 0

        for X_batch, y_batch in train_loader:
            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device).unsqueeze(1)

            optimizer.zero_grad()
            outputs = model(X_batch)
            loss = criterion(outputs, y_batch)
            loss.backward()
            optimizer.step()

            total_loss += loss.item()

        print(f"Epoch {epoch+1}, Loss: {total_loss/len(train_loader)}")

        evaluate(model, test_loader)

In [37]:
from sklearn.metrics import accuracy_score, classification_report
import numpy as np

def evaluate(model, loader):
    model.eval()

    all_preds = []
    all_labels = []

    with torch.no_grad():
        for X_batch, y_batch in loader:

            X_batch = X_batch.to(device)
            y_batch = y_batch.to(device)

            outputs = model(X_batch)

            # Convert logits → probabilities
            probs = torch.sigmoid(outputs)

            # Convert probabilities → 0/1
            preds = (probs > 0.5).float()

            # Flatten tensors before storing
            all_preds.extend(preds.cpu().numpy().ravel())
            all_labels.extend(y_batch.cpu().numpy().ravel())

    # Convert to numpy arrays
    all_preds = np.array(all_preds)
    all_labels = np.array(all_labels)

    # Accuracy
    acc = accuracy_score(all_labels, all_preds)
    print("\nTest Accuracy:", round(acc * 100, 2), "%")

    # Classification Report
    print("\nClassification Report:\n")
    print(classification_report(all_labels, all_preds, digits=4))

In [39]:
import torch
import torch.nn as nn

class LSTM_OneLayer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_matrix):
        super(LSTM_OneLayer, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=False  # allow fine-tuning
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=1,
            batch_first=True
        )

        self.dropout = nn.Dropout(0.5)
        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)

        out = self.dropout(hidden[-1])
        out = self.fc(out)

        return out

In [40]:
class LSTM_TwoLayer(nn.Module):
    def __init__(self, vocab_size, embedding_dim, hidden_dim, embedding_matrix):
        super(LSTM_TwoLayer, self).__init__()

        self.embedding = nn.Embedding.from_pretrained(
            embedding_matrix,
            freeze=False
        )

        self.lstm = nn.LSTM(
            input_size=embedding_dim,
            hidden_size=hidden_dim,
            num_layers=2,
            batch_first=True,
            dropout=0.5
        )

        self.fc = nn.Linear(hidden_dim, 1)

    def forward(self, x):
        embedded = self.embedding(x)
        _, (hidden, _) = self.lstm(embedded)

        out = hidden[-1]
        out = self.fc(out)

        return out

In [41]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")


LSTM One Layer

In [42]:
hidden_dim = 128

model = LSTM_OneLayer(vocab_size, embedding_dim, hidden_dim, embedding_matrix)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)
train_model(model, train_loader, test_loader, epochs=10)

Epoch 1, Loss: 0.6066059052182463

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2, Loss: 0.604885821239636

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 3, Loss: 0.6053403459798795

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 9, Loss: 0.6031885987843949

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 10, Loss: 0.6046416560491434

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


LSTM Two Layers

In [43]:
hidden_dim = 128

model = LSTM_TwoLayer(vocab_size, embedding_dim, hidden_dim, embedding_matrix)
model = model.to(device)

criterion = nn.BCEWithLogitsLoss()
optimizer = torch.optim.Adam(model.parameters(), lr=0.001)

train_model(model, train_loader, test_loader, epochs=10)

Epoch 1, Loss: 0.6058866623491524

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 2, Loss: 0.6035612628292352

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 3, Loss: 0.6037584545132452

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



...

/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 9, Loss: 0.603003051524726

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))


Epoch 10, Loss: 0.6023206444213185

Test Accuracy: 70.97 %

Classification Report:

              precision    recall  f1-score   support

         0.0     0.7097    1.0000    0.8302      2364
         1.0     0.0000    0.0000    0.0000       967

    accuracy                         0.7097      3331
   macro avg     0.3548    0.5000    0.4151      3331
weighted avg     0.5037    0.7097    0.5892      3331



/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
/usr/local/lib/python3.12/dist-packages/sklearn/metrics/_classification.py:1565: UndefinedMetricWarning: Precision is ill-defined and being set to 0.0 in labels with no predicted samples. Use `zero_division` parameter to control this behavior.
  _warn_prf(average, modifier, f"{metric.capitalize()} is", len(result))
